In [0]:
# Tarea 4: Registro de características en el Feature Store
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql.functions import col, lit

# 1. Inicializar el cliente de Feature Engineering
fe = FeatureEngineeringClient()

# 2. Configuración de nombres (Uso de Unity Catalog)
# Ajusta 'main' y 'ml_production' a tus nombres reales
CATALOG = "workspace"
SCHEMA = "gold"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.store_performance_features"

# 3. Preparar el DataFrame de características desde la Capa Gold
# Leemos los datos que ya procesamos en la tarea anterior
gold_df = spark.read.table(f"{CATALOG}.gold.regional_sales_performance")

# Seleccionamos y renombramos columnas para que sean "Features" claras
# Agregamos una columna de 'update_timestamp' para control de versiones
features_df = gold_df.select(
    col("store_id"),
    col("sales_date"),
    col("total_transactions").alias("feat_rolling_transactions"),
    col("total_revenue").alias("feat_rolling_revenue"),
    col("total_items_sold").alias("feat_items_sold")
)

# 4. Creación o actualización de la Feature Table
# Si la tabla no existe, la crea. Si existe, la actualiza.
try:
    # Intentamos crear la tabla (esto solo se ejecuta exitosamente la primera vez)
    fe.create_table(
        name=TABLE_NAME,
        primary_keys=["store_id", "sales_date"],
        timeseries_columns=["sales_date"],
        df=features_df,
        description="Características históricas de ventas por tienda para predicción de demanda en Walmart."
    )
    print(f"Feature Table {TABLE_NAME} creada exitosamente.")
except Exception as e:
    # Si ya existe, simplemente escribimos los nuevos datos (Upsert)
    print("La tabla ya existe. Realizando carga incremental de características...")
    fe.write_table(
        name=TABLE_NAME,
        df=features_df,
        mode="merge"
    )

# 5. Linaje de Datos
print(f"Proceso terminado. Las características están disponibles en Unity Catalog bajo: {TABLE_NAME}")